# Curvature-Coupled Dark Energy: Effective Poisson Function and Linear Growth

This notebook reproduces **paper Fig. 6**.

The left panel shows $\mu_\Psi(k,z)$ reconstructed from the full linear perturbation
evolution at $k=10\,{\rm Mpc}^{-1}$. The right panel shows
\[
f(k,z)=\frac{d\ln|\delta_m|}{d\ln a}.
\]

The same representative models, colours, and line-width convention are used as in the
other CCDE notebooks.


In [ ]:

import os
from os.path import exists

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# -------------------------------------------------
# Paper-wide plotting convention
# -------------------------------------------------

text_size  = 30
fig_size_x = 19
fig_size_y = 7
label_fs   = 34

lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

major_alpha = 0.35
minor_alpha = 0.15

colors = [
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = [
    "0.001", "0.1", "0.001", "-0.1",
    "0.1", "-0.1", "0.1", "0.001", "-0.05"
]

sigma_consts = [
    "0.001", "0.3", "0.3", "0.3",
    "1.", "1.", "1.5", "1.5", "1.5"
]

models = list(zip(sigma_consts, alpha_consts))


## 1. Load CCDE background and perturbation outputs

The data are read from `../DataGenerator/transfer_functions/` using the renamed CCDE files.
The original dictionary structure is retained:

`data["bg_data"][sigma_key][alpha_key]`  
`data["perturbations"][sigma_key][alpha_key][k_index]`


In [ ]:

# -------------------------------------------------
# Configuration
# -------------------------------------------------
data_address = "./../DataGenerator/transfer_functions/"
load_data = True

alphas = ["0.001", "0.05", "0.1", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]

# Original physical k outputs [Mpc^{-1}].
# CLASS writes the perturbation files in increasing-k order.
k_indices = list(range(9))
k_output_values = [
    0.0001,
    0.005,
    0.001,
    0.05,
    0.01,
    0.5,
    0.1,
    1.0,
    10.0,
]

k_output_values_sorted = sorted(k_output_values)

k_index_to_value = {
    k_ind: k_val
    for k_ind, k_val in zip(k_indices, k_output_values_sorted)
}


def skey(s):
    return f"sigma={s}"


def akey(a):
    return f"alpha={a}"


alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

out_arr = np.empty((len(sigma), len(alpha)), dtype=object)

# Preserve the original data construction.
data = {
    "perturbations": {},
    "bg_data": {},
}

for s_str in sigma:
    data["perturbations"].setdefault(skey(s_str), {})
    data["bg_data"].setdefault(skey(s_str), {})

    for a_str in alpha:
        data["perturbations"][skey(s_str)].setdefault(akey(a_str), {})
        data["bg_data"][skey(s_str)].setdefault(akey(a_str), None)


# -------------------------------------------------
# Load files
# -------------------------------------------------
total_loaded_perturbations = 0
total_loaded_backgrounds = 0
missing_perturbations = []
missing_backgrounds = []

if load_data:

    for a_str in alpha:
        for s_str in sigma:

            output_dir = f"sigma{s_str}_alpha{a_str}"
            base_path = os.path.join(
                data_address,
                "run_" + output_dir,
            )

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i, j] = output_dir

            bg_file = os.path.join(
                base_path,
                f"CCDE_sigma{s_str}_alpha{a_str}_background.dat",
            )

            if exists(bg_file):
                data["bg_data"][skey(s_str)][akey(a_str)] = np.loadtxt(bg_file)
                total_loaded_backgrounds += 1
            else:
                data["bg_data"][skey(s_str)][akey(a_str)] = None
                missing_backgrounds.append(bg_file)

            for k_ind in k_indices:
                pert_file = os.path.join(
                    base_path,
                    (
                        f"CCDE_sigma{s_str}_alpha{a_str}"
                        f"_perturbations_k{k_ind}_s.dat"
                    ),
                )

                if exists(pert_file):
                    data["perturbations"][skey(s_str)][akey(a_str)][k_ind] = (
                        np.loadtxt(pert_file)
                    )
                    total_loaded_perturbations += 1
                else:
                    data["perturbations"][skey(s_str)][akey(a_str)][k_ind] = None
                    missing_perturbations.append(pert_file)
            print(f"\033[94m{output_dir} exists — loaded\033[0m")

print("Background files loaded:", total_loaded_backgrounds)
print("Perturbation files loaded:", total_loaded_perturbations)
print("k-index mapping:", k_index_to_value)

if missing_backgrounds:
    print(f"Missing {len(missing_backgrounds)} background files.")

if missing_perturbations:
    print(f"Missing {len(missing_perturbations)} perturbation files.")


## 2. Construct $\mu_\Psi$ and the growth rate

The Poisson response uses the gauge-invariant comoving matter perturbation,
$
\mu_\Psi
=
-\frac{k^2\Psi}
{4\pi G\,a^2\rho_m\Delta_m}.
$

The growth rate follows the paper definition,
$
f(k,z)=d\ln|\delta_m|/d\ln a.
$
Thus $\Delta_m$ is used for the Poisson equation, while $\delta_m$ is used for growth.


In [ ]:

# -------------------------------------------------
# Perturbation columns (zero-based)
# -------------------------------------------------
PERT_COL = {
    "tau": 0,
    "a": 1,
    "delta_b": 8,
    "theta_b": 9,
    "psi": 10,
    "phi": 11,
    "delta_cdm": 15,
    "theta_cdm": 16,
    "delta_ncdm": 17,
    "theta_ncdm": 18,
}

# -------------------------------------------------
# CCDE background columns (zero-based)
# -------------------------------------------------
BG_COL = {
    "z": 0,
    "H": 3,
    "rho_b": 9,
    "rho_cdm": 10,
    "rho_ncdm": 11,
    "p_ncdm": 12,
}


def prepare_perturbation_series(arr):
    arr = np.asarray(arr, dtype=float)

    valid = (
        np.isfinite(arr[:, PERT_COL["tau"]])
        & np.isfinite(arr[:, PERT_COL["a"]])
        & (arr[:, PERT_COL["a"]] > 0.0)
    )

    arr = arr[valid]
    arr = arr[np.argsort(arr[:, PERT_COL["a"]])]

    a = arr[:, PERT_COL["a"]]
    _, unique_indices = np.unique(a, return_index=True)

    return arr[np.sort(unique_indices)]


def prepare_background_series(bg_arr):
    bg_arr = np.asarray(bg_arr, dtype=float)

    z = bg_arr[:, BG_COL["z"]]
    a = 1.0 / (1.0 + z)

    valid = np.isfinite(z) & np.isfinite(a) & (a > 0.0)

    bg_arr = bg_arr[valid]
    a = a[valid]

    order = np.argsort(a)
    bg_arr = bg_arr[order]
    a = a[order]

    _, unique_indices = np.unique(a, return_index=True)

    return bg_arr[np.sort(unique_indices)]


def interpolate_background_to_a(bg_arr, a_target, column):
    bg_arr = prepare_background_series(bg_arr)

    z_bg = bg_arr[:, BG_COL["z"]]
    a_bg = 1.0 / (1.0 + z_bg)
    y_bg = bg_arr[:, column]

    valid = np.isfinite(a_bg) & np.isfinite(y_bg)

    return np.interp(
        a_target,
        a_bg[valid],
        y_bg[valid],
        left=np.nan,
        right=np.nan,
    )


def logarithmic_derivative(y, a):
    y = np.asarray(y, dtype=float)
    a = np.asarray(a, dtype=float)

    result = np.full_like(y, np.nan)

    finite_y = np.abs(y[np.isfinite(y)])

    if finite_y.size == 0:
        return result

    scale = np.nanmax(finite_y)

    valid = (
        np.isfinite(y)
        & np.isfinite(a)
        & (a > 0.0)
        & (np.abs(y) > 1.e-12 * scale)
    )

    if np.count_nonzero(valid) < 5:
        return result

    result[valid] = np.gradient(
        np.log(np.abs(y[valid])),
        np.log(a[valid]),
        edge_order=2,
    )

    return result


def matter_perturbations(pert_arr, bg_arr, k_Mpc):
    pert_arr = prepare_perturbation_series(pert_arr)

    a = pert_arr[:, PERT_COL["a"]]
    z = 1.0 / a - 1.0

    H_phys = interpolate_background_to_a(bg_arr, a, BG_COL["H"])
    Hcal = a * H_phys

    rho_b = interpolate_background_to_a(bg_arr, a, BG_COL["rho_b"])
    rho_cdm = interpolate_background_to_a(bg_arr, a, BG_COL["rho_cdm"])
    rho_ncdm = interpolate_background_to_a(bg_arr, a, BG_COL["rho_ncdm"])
    p_ncdm = interpolate_background_to_a(bg_arr, a, BG_COL["p_ncdm"])

    delta_b = pert_arr[:, PERT_COL["delta_b"]]
    theta_b = pert_arr[:, PERT_COL["theta_b"]]

    delta_cdm = pert_arr[:, PERT_COL["delta_cdm"]]
    theta_cdm = pert_arr[:, PERT_COL["theta_cdm"]]

    delta_ncdm = pert_arr[:, PERT_COL["delta_ncdm"]]
    theta_ncdm = pert_arr[:, PERT_COL["theta_ncdm"]]

    psi = pert_arr[:, PERT_COL["psi"]]

    rho_m = rho_b + rho_cdm + rho_ncdm

    delta_rho_m = (
        rho_b * delta_b
        + rho_cdm * delta_cdm
        + rho_ncdm * delta_ncdm
    )

    delta_m = delta_rho_m / rho_m

    momentum_m = (
        rho_b * theta_b
        + rho_cdm * theta_cdm
        + (rho_ncdm + p_ncdm) * theta_ncdm
    )

    Delta_m = (
        delta_rho_m
        + 3.0 * Hcal * momentum_m / k_Mpc**2
    ) / rho_m

    return {
        "a": a,
        "z": z,
        "rho_m": rho_m,
        "delta_m": delta_m,
        "Delta_m": Delta_m,
        "psi": psi,
    }


def compute_mu_and_growth(pert_arr, bg_arr, k_Mpc):
    q = matter_perturbations(
        pert_arr,
        bg_arr,
        k_Mpc,
    )

    a = q["a"]
    rho_m = q["rho_m"]
    Delta_m = q["Delta_m"]
    delta_m = q["delta_m"]
    psi = q["psi"]

    # CLASS background densities are stored as (8 pi G / 3) rho.
    denominator = 1.5 * a**2 * rho_m * Delta_m

    mu_psi = np.full_like(psi, np.nan, dtype=float)

    finite_den = np.abs(
        denominator[np.isfinite(denominator)]
    )

    if finite_den.size > 0:
        scale = np.nanmax(finite_den)

        safe = (
            np.isfinite(denominator)
            & np.isfinite(psi)
            & (np.abs(denominator) > 1.e-14 * scale)
        )

        mu_psi[safe] = (
            -k_Mpc**2 * psi[safe]
            / denominator[safe]
        )

    # Paper Fig. 6 uses delta_m for the growth rate.
    growth_rate = logarithmic_derivative(
        delta_m,
        a,
    )

    q["mu_psi"] = mu_psi
    q["growth_rate"] = growth_rate

    return q


## 3. Effective Poisson function and linear growth rate — paper Fig. 6

The plotted mode is $k=10\,{\rm Mpc}^{-1}$, where $\mu_\Psi$ has reached its
quasistatic high-$k$ limit. The near-$\Lambda$CDM case
$(\sigma,\alpha)=(0.001,0.001)$ is shown in black.


In [ ]:
# ============================================================
# WAVENUMBER
# ============================================================

k_target = 10.0

k_ind = min(
    k_index_to_value,
    key=lambda i: abs(
        k_index_to_value[i] - k_target
    )
)

k_actual = k_index_to_value[k_ind]

print(
    f"Using k-index {k_ind}, corresponding to "
    f"k={k_actual:g} Mpc^(-1)"
)


# ============================================================
# FIGURE: 1 row x 2 columns
# ============================================================

fig, (ax_mu, ax_f) = plt.subplots(
    1,
    2,
    figsize=(fig_size_x, fig_size_y),
    sharex=True,
    facecolor='white',
    gridspec_kw={
        "wspace": 0.25,
    }
)

all_axes = [ax_mu, ax_f]

for ax in all_axes:

    ax.tick_params(
        which='both',
        direction='in',
        top=True,
        right=True
    )

    ax.grid(
        True,
        which='major',
        alpha=major_alpha
    )

    ax.grid(
        True,
        which='minor',
        alpha=minor_alpha
    )

    ax.minorticks_on()


# ============================================================
# PLOT MODELS
# ============================================================

for num, (sigma_string, alpha_string) in enumerate(models):

    sigma_key = skey(sigma_string)
    alpha_key = akey(alpha_string)

    pert_arr = (
        data["perturbations"]
        .get(sigma_key, {})
        .get(alpha_key, {})
        .get(k_ind, None)
    )

    bg_arr = (
        data["bg_data"]
        .get(sigma_key, {})
        .get(alpha_key, None)
    )

    if not isinstance(pert_arr, np.ndarray):

        print(
            f"Skipping sigma={sigma_string}, "
            f"alpha={alpha_string}: perturbation file missing."
        )

        continue

    if not isinstance(bg_arr, np.ndarray):

        print(
            f"Skipping sigma={sigma_string}, "
            f"alpha={alpha_string}: background file missing."
        )

        continue


    # Reconstruct mu_Psi from the comoving matter perturbation
    # and compute the growth rate from delta_m.
    result = compute_mu_and_growth(
        pert_arr,
        bg_arr,
        k_actual
    )

    x = 1.0 + result["z"]

    mu_psi = result["mu_psi"]
    growth_rate = result["growth_rate"]


    # Remove undefined/non-finite samples from the plotted curves.
    valid_mu = (
        np.isfinite(x)
        & np.isfinite(mu_psi)
        & (x > 0.0)
    )

    valid_f = (
        np.isfinite(x)
        & np.isfinite(growth_rate)
        & (x > 0.0)
    )


    sigma_value = float(sigma_string)
    alpha_value = float(alpha_string)

    label = (
        rf'$(\sigma={sigma_value:g},\,'
        rf'\alpha={alpha_value:g})$'
    )

    color = colors[num]


    # --------------------------------------------------------
    # Left panel: effective Poisson function mu_Psi
    # --------------------------------------------------------

    ax_mu.plot(
        x[valid_mu],
        mu_psi[valid_mu],
        '-',
        lw=lw_f,
        c=color
    )


    # --------------------------------------------------------
    # Right panel: linear matter growth rate
    # --------------------------------------------------------

    ax_f.plot(
        x[valid_f],
        growth_rate[valid_f],
        '-',
        lw=lw_f,
        c=color,
        label=label
    )


# ============================================================
# LABELS
# ============================================================

ax_mu.set_ylabel(
    rf'$\mu_\Psi(k={k_actual:g}\,'
    rf'{{\rm Mpc}}^{{-1}},z)$',
    fontsize=label_fs
)

ax_f.set_ylabel(
    rf'$f(k={k_actual:g}\,'
    rf'{{\rm Mpc}}^{{-1}},z)$',
    fontsize=label_fs
)

ax_mu.set_xlabel(
    r'$1+z$',
    fontsize=label_fs
)

ax_f.set_xlabel(
    r'$1+z$',
    fontsize=label_fs
)


# ============================================================
# AXIS LIMITS
# ============================================================

for ax in all_axes:

    ax.set_xlim(
        1.0,
        100.0
    )

    ax.set_xscale(
        'log'
    )

ax_mu.set_ylim(
    0.75,
    1.95
)

ax_f.set_ylim(
    0.45,
    1.20
)


# ============================================================
# TICKS
# ============================================================

ax_mu.yaxis.set_major_locator(
    MultipleLocator(0.2)
)

ax_mu.yaxis.set_minor_locator(
    MultipleLocator(0.05)
)

ax_f.yaxis.set_major_locator(
    MultipleLocator(0.1)
)

ax_f.yaxis.set_minor_locator(
    MultipleLocator(0.025)
)


# ============================================================
# LEGEND
# ============================================================

leg = ax_f.legend(
    loc='upper left',
    bbox_to_anchor=(0.15, 0.40),
    frameon=True,
    fontsize=15.5,
    ncol=2,
    columnspacing=0.3
)

leg.get_frame().set_alpha(
    0.85
)


# ============================================================
# SAVE AND SHOW
# ============================================================

# plt.tight_layout()

plt.savefig(
    './Figs/mu_growth_k10.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()